# Otimização — capítulo 2: Otimização unidimensional
### Exemplos em Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/optimization-course/blob/main/code/python/notebooks/02_univariate_optimization.ipynb)

Cada secção corre o exemplo `ex….py` do método e compara os resultados com os slides (no fim: *confere com os slides: sim*). Comece pela célula **Preparação**; no Colab, *Runtime → Run all* corre tudo.

Os slides usam vírgula decimal; aqui usa-se o ponto. Este caderno é gerado a partir dos ficheiros `ex….py` por `make_notebooks.py`.

*Examples for Chapter 2 of the course "Optimization" (ISEL, J. F. A. Madeira). Code comments and printed messages are in Portuguese; the English slides are in [`notes/en/`](https://github.com/aguilarmadeira/optimization-course/tree/main/notes/en).*

## Preparação

In [1]:
# Preparação: põe as funções da UC no caminho do Python.
# No Colab (ou noutra pasta), clona o repositório; dentro do repositório, usa code/python.
import os, sys
LOCAL = [p for p in ("..", os.path.join("..", "..")) if os.path.isfile(os.path.join(p, "uc_setup.py"))]
if LOCAL:
    RAIZ = os.path.abspath(LOCAL[0])
else:
    if not os.path.isdir("optimization-course"):
        !git clone -q --depth 1 https://github.com/aguilarmadeira/optimization-course
    RAIZ = os.path.abspath(os.path.join("optimization-course", "code", "python"))
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import uc_setup  # acrescenta as pastas dos métodos e common/
print("Código da UC pronto.")

Código da UC pronto.


## 2.2 Método da secção áurea

Exemplo [`ex02_2_golden_section.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/02_univariate_optimization/02_2_golden_section/ex02_2_golden_section.py) · funções da UC: [`golden_section.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/02_univariate_optimization/02_2_golden_section/golden_section.py)

```text
Reproduz os exemplos do deck 2.2 (secção áurea).

Exemplo 2 (exemplo-guia): f(x) = x^2 + 54/x em [1,4], 15 reduções.
Exemplo 1:                f(x) = -3 sin(x) + x^2/8 em [0,4], 15 reduções.
Slide «Quantas reduções?»: L0 = 1, tolx = 0.01  =>  N = 10, n_f = 12.

Imprime as tabelas das iterações (k = 0..15; os slides mostram k = 0..7),
os valores finais e as contagens, e no fim compara com os slides.
Os slides usam vírgula decimal; aqui usa-se o ponto.
```

In [2]:
import numpy as np

from golden_section import golden_section


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


ok = True
print("Otimização — deck 2.2: método da secção áurea")

# ------------------------------------------------------------ Exemplo 2
f = lambda x: x**2 + 54 / x
a, b, N = 1, 4, 15
print("\nExemplo 2: f(x) = x^2 + 54/x em [%g,%g], N = %d reduções" % (a, b, N))
r = golden_section(f, a, b, N, verbose=True)
L15 = r.history[-1, 7]
print("Após %d reduções: L_%d = %.4f, x* ~ %.4f, f(x*) ~ %.4f  (exato: x* = 3, f = 27)"
      % (N, N, L15, r.x, r.fx))
print("n_f = N + 2 = %d para localizar x*; com f no ponto médio final, n_f = N + 3 = %d"
      % (r.nfev_loc, r.nfev))

# tabela do slide (k = 0..7): a  b  x1  x2  f(x1)  f(x2)  L_k
T2 = [[1.0000, 4.0000, 2.1459, 2.8541, 29.7692, 27.0660, 3.0000],
      [2.1459, 4.0000, 2.8541, 3.2918, 27.0660, 27.2403, 1.8541],
      [2.1459, 3.2918, 2.5836, 2.8541, 27.5761, 27.0660, 1.1459],
      [2.5836, 3.2918, 2.8541, 3.0213, 27.0660, 27.0014, 0.7082],
      [2.8541, 3.2918, 3.0213, 3.1246, 27.0014, 27.0453, 0.4377],
      [2.8541, 3.1246, 2.9574, 3.0213, 27.0055, 27.0014, 0.2705],
      [2.9574, 3.1246, 3.0213, 3.0608, 27.0014, 27.0109, 0.1672],
      [2.9574, 3.0608, 2.9969, 3.0213, 27.0000, 27.0014, 0.1033]]
c = [confere(r.history[:8, 1:8], T2, 4), confere(L15, 0.0022, 4),
     confere(r.x, 3.0002, 4), confere(r.fx, 27.0000, 4),
     r.nfev_loc == 17, r.nfev == 18]
print("  tabela k = 0..7: %s | L_15: %s | x*: %s | f(x*): %s | n_f = 17: %s | n_f = 18: %s"
      % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ Exemplo 1
f = lambda x: -3 * np.sin(x) + x**2 / 8
a, b, N = 0, 4, 15
print("\nExemplo 1: f(x) = -3 sin(x) + x^2/8 em [%g,%g], N = %d reduções" % (a, b, N))
r = golden_section(f, a, b, N, verbose=True)
L15 = r.history[-1, 7]
print("Após %d reduções: L_%d = %.4f, x* ~ %.4f, f(x*) ~ %.4f;  n_f = %d (+1 = %d)"
      % (N, N, L15, r.x, r.fx, r.nfev_loc, r.nfev))
T1 = [[0.0000, 4.0000, 1.5279, 2.4721, -2.7054, -1.0977, 4.0000],
      [0.0000, 2.4721, 0.9443, 1.5279, -2.3188, -2.7054, 2.4721],
      [0.9443, 2.4721, 1.5279, 1.8885, -2.7054, -2.4040, 1.5279],
      [0.9443, 1.8885, 1.3050, 1.5279, -2.6818, -2.7054, 0.9443],
      [1.3050, 1.8885, 1.5279, 1.6656, -2.7054, -2.6397, 0.5836],
      [1.3050, 1.6656, 1.4427, 1.5279, -2.7152, -2.7054, 0.3607],
      [1.3050, 1.5279, 1.3901, 1.4427, -2.7096, -2.7152, 0.2229],
      [1.3901, 1.5279, 1.4427, 1.4752, -2.7152, -2.7143, 0.1378]]
c = [confere(r.history[:8, 1:8], T1, 4), confere(L15, 0.0029, 4),
     confere(r.x, 1.4489, 4), confere(r.fx, -2.7153, 4)]
print("  tabela k = 0..7: %s | L_15: %s | x*: %s | f(x*): %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ «Quantas reduções?»
tau = (np.sqrt(5) - 1) / 2
L0, tolx = 1, 0.01
Nmin = np.log(tolx / L0) / np.log(tau)
N = int(np.ceil(Nmin))
r = golden_section(lambda x: (x - 0.3)**2, 0, L0, N)
print("\nQuantas reduções? L0 = %g, tolx = %g: N >= %.2f => N = %d, tau^N = %.4f, n_f = %d"
      % (L0, tolx, Nmin, N, tau**N, r.nfev_loc))
c = [confere(Nmin, 9.57, 2), N == 10, confere(tau**N, 0.0081, 4), r.nfev_loc == 12]
print("  N >= 9.57: %s | N = 10: %s | 0.618^10: %s | n_f = 12: %s" % simnao(c))
ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 2.2: método da secção áurea

Exemplo 2: f(x) = x^2 + 54/x em [1,4], N = 15 reduções
  k         a         b        x1        x2      f(x1)      f(x2)       L_k
  0    1.0000    4.0000    2.1459    2.8541    29.7692    27.0660    3.0000
  1    2.1459    4.0000    2.8541    3.2918    27.0660    27.2403    1.8541
  2    2.1459    3.2918    2.5836    2.8541    27.5761    27.0660    1.1459
  3    2.5836    3.2918    2.8541    3.0213    27.0660    27.0014    0.7082
  4    2.8541    3.2918    3.0213    3.1246    27.0014    27.0453    0.4377
  5    2.8541    3.1246    2.9574    3.0213    27.0055    27.0014    0.2705
  6    2.9574    3.1246    3.0213    3.0608    27.0014    27.0109    0.1672
  7    2.9574    3.0608    2.9969    3.0213    27.0000    27.0014    0.1033
  8    2.9574    3.0213    2.9818    2.9969    27.0010    27.0000    0.0639
  9    2.9818    3.0213    2.9969    3.0062    27.0000    27.0001    0.0395
 10    2.9818    3.0062    2.9911    2.9969    27.0002    27.0

## 2.3 Interpolação quadrática sucessiva (método de Powell)

Exemplo [`ex02_3_powell.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/02_univariate_optimization/02_3_powell/ex02_3_powell.py) · funções da UC: [`powell_quadratic.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/02_univariate_optimization/02_3_powell/powell_quadratic.py)

```text
Reproduz os exemplos do deck 2.3 (interpolação quadrática sucessiva).

Exemplo: f(x) = x^2 + 54/x, x1 = 1, Delta = 1, tolx = tolf = 1e-3:
         4 iterações (k = 0..3), sem reflexões, n_f = 3 + 4 = 7, x* = 3.
«Quando a parábola não serve»: f(x) = x^4/4 - x^2/2 com os trios
         (0.3, 0.45, 0.6) -> a2 < 0, xb = -0.46;
         (0.55, 0.6, 0.65) -> a2 ~ 0+, xb = 5.31, f(xb) = 184;
         e a salvaguarda (reflexão) a partir do primeiro trio.

Imprime a tabela das iterações como nos slides, os valores finais e as
contagens, e no fim compara com os slides (vírgula decimal nos slides,
ponto aqui).  Correr (de qualquer pasta):  python ex02_3_powell.py
```

In [3]:
import numpy as np

from powell_quadratic import powell_quadratic


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


ok = True
print("Otimização — deck 2.3: interpolação quadrática sucessiva (Powell)")

# ------------------------------------------------------------ exemplo
f = lambda x: x**2 + 54 / x
x1, Delta, tolx, tolf, kmax = 1, 1, 1e-3, 1e-3, 50
print("\nExemplo: f(x) = x^2 + 54/x, x1 = %g, Delta = %g, tolx = tolf = %g" % (x1, Delta, tolx))
r = powell_quadratic(f, x1, Delta, tolx, tolf, kmax, verbose=True)
print("Fim: x* = %.4f, f(x*) = %.4f; %d iterações, %d reflexões, n_f = 3 + %d = %d"
      % (r.x, r.fx, r.nit, r.nrefl, r.nit + r.nrefl, r.nfev))
print("Erros |xb - 3|:" + "".join(" %.1e" % e for e in np.abs(r.history[:, 9] - 3)))

# tabela do slide: x1 x2 x3 (4 casas) f1 f2 f3 (4) a1 a2 (3) xb f(xb) (4)
T = np.array([[1.0000, 2.0000, 3.0000, 55.0000, 31.0000, 27.0000, -24.000, 10.000, 2.7000, 27.2900],
              [2.0000, 2.7000, 3.0000, 31.0000, 27.2900, 27.0000, -5.300, 4.333, 2.9615, 27.0045],
              [2.7000, 2.9615, 3.0000, 27.2900, 27.0045, 27.0000, -1.092, 3.251, 2.9987, 27.0000],
              [2.9615, 2.9987, 3.0000, 27.0045, 27.0000, 27.0000, -0.120, 3.027, 3.0000, 27.0000]])
H = r.history
tabok = (H.shape[0] == 4
         and confere(H[:, [1, 2, 3, 4, 5, 6, 9, 10]], T[:, [0, 1, 2, 3, 4, 5, 8, 9]], 4)
         and confere(H[:, 7:9], T[:, 6:8], 3))
erros = np.abs(H[:, 9] - 3)
c = [tabok, r.nit == 4, r.nrefl == 0, r.nfev == 7,
     confere(r.x, 3, 4), confere(r.fx, 27, 4),
     confere(erros[1:4], [0.04, 0.0013, 6e-6], [2, 4, 6])]
print("  tabela k = 0..3: %s | 4 it.: %s | sem reflexões: %s | n_f = 7: %s | x* = 3: %s | f = 27: %s | erros: %s"
      % simnao(c))
ok = ok and all(c)

# ------------------------------------------ quando a parábola não serve
g = lambda x: x**4 / 4 - x**2 / 2
print("\nQuando a parábola não serve: f(x) = x^4/4 - x^2/2 (mínimos em -1 e 1)")
P = [[0.3, 0.45, 0.6], [0.55, 0.6, 0.65]]
xbP = []; a2P = []
for X in P:
    F = [g(xi) for xi in X]
    a1 = (F[1] - F[0]) / (X[1] - X[0])
    a2 = ((F[2] - F[0]) / (X[2] - X[0]) - a1) / (X[2] - X[1])
    xb = (X[0] + X[1]) / 2 - a1 / (2 * a2)
    xbP.append(xb); a2P.append(a2)
    print("  trio (%.2f, %.2f, %.2f): a1 = %.4f, a2 = %.4f, xb = %.2f, f(xb) = %.1f"
          % (*X, a1, a2, xb, g(xb)))
rr = powell_quadratic(g, 0.3, 0.15, 1e-3, 1e-3, 50)
print("  com a salvaguarda, de x1 = 0.3, Delta = 0.15: x* = %.4f, f = %.4f, %d reflexão(ões), %d it., n_f = %d"
      % (rr.x, rr.fx, rr.nrefl, rr.nit, rr.nfev))
c = [a2P[0] < 0, confere(xbP[0], -0.46, 2), a2P[1] > 0, confere(xbP[1], 5.31, 2),
     confere(g(xbP[1]), 184, 0), rr.flag == 0, confere(rr.x, 1, 2)]
print("  a2 < 0: %s | xb = -0.46: %s | a2 > 0: %s | xb = 5.31: %s | f(xb) = 184: %s | reflexão converge: %s | x* ~ 1: %s"
      % simnao(c))
ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 2.3: interpolação quadrática sucessiva (Powell)

Exemplo: f(x) = x^2 + 54/x, x1 = 1, Delta = 1, tolx = tolf = 0.001
  k        x1        x2        x3         f1         f2         f3        a1       a2        xb      f(xb)
  0    1.0000    2.0000    3.0000    55.0000    31.0000    27.0000   -24.000   10.000    2.7000    27.2900
  1    2.0000    2.7000    3.0000    31.0000    27.2900    27.0000    -5.300    4.333    2.9615    27.0045
  2    2.7000    2.9615    3.0000    27.2900    27.0045    27.0000    -1.092    3.251    2.9987    27.0000
  3    2.9615    2.9987    3.0000    27.0045    27.0000    27.0000    -0.120    3.027    3.0000    27.0000
Fim: x* = 3.0000, f(x*) = 27.0000; 4 iterações, 0 reflexões, n_f = 3 + 4 = 7
Erros |xb - 3|: 3.0e-01 3.8e-02 1.3e-03 5.7e-06
  tabela k = 0..3: sim | 4 it.: sim | sem reflexões: sim | n_f = 7: sim | x* = 3: sim | f = 27: sim | erros: sim

Quando a parábola não serve: f(x) = x^4/4 - x^2/2 (mínimos em -1 e 1)
  trio (0.30, 0.45, 0.

## 2.4 Método da bisseção

Exemplo [`ex02_4_bisection.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/02_univariate_optimization/02_4_bisection/ex02_4_bisection.py) · funções da UC: [`bisection.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/02_univariate_optimization/02_4_bisection/bisection.py)

```text
Reproduz os exemplos do deck 2.4 (bisseção).

Exemplo 2 (exemplo-guia): f(x) = x^2 + 54/x em [1,4], 15 reduções:
          x ~ 3.0000, b - a = 3/2^15 = 9.2e-5, n_g = N + 2 = 17, n_f = 0.
Exemplo 1: f(x) = -2 sin(x) + x^2/16 em [-1,3], 15 reduções.
«Para resolver»: x + 31/(x+3) em [0,5] e -3 sin(x) + x^2/8 em [0,4].
«Quantas iterações?»: [0,2], tolx = 1e-3 -> k = 11; tolx = 1e-4 -> k = 15.

Imprime as tabelas das iterações (k = 0..14; os slides mostram k = 0..7),
os valores finais e as contagens, e no fim compara com os slides
(vírgula decimal nos slides, ponto aqui). A coluna f(x_k) é só para
leitura: o método não a usa.
```

In [4]:
import numpy as np

from bisection import bisection


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


ok = True
print("Otimização — deck 2.4: bisseção")

# ------------------------------------------------------------ Exemplo 2
f = lambda x: x**2 + 54 / x
df = lambda x: 2 * x - 54 / x**2
a, b, N = 1, 4, 15
print("\nExemplo 2: f(x) = x^2 + 54/x em [%g,%g], f'(%g) = %g < 0, f'(%g) = %g > 0, N = %d"
      % (a, b, a, df(a), b, df(b), N))
r = bisection(f, df, a, b, N, verbose=True)
print("Após %d reduções, ponto médio: x* ~ %.4f, f(x*) ~ %.4f, b - a = %.1e (exato: x* = 3)"
      % (N, r.x, r.fx, r.L))
print("n_g = N + 2 = %d, n_f = %d" % (r.ngev, r.nfev))
tau = (np.sqrt(5) - 1) / 2; Lgs = 3 * tau**15
print("Secção áurea, mesmo [1,4], 15 reduções: L = %.4f, %.0f vezes maior" % (Lgs, Lgs / r.L))
T2 = [[1.0000, 4.0000, 2.5000, 27.8500, -3.6400],
      [2.5000, 4.0000, 3.2500, 27.1779, 1.3876],
      [2.5000, 3.2500, 2.8750, 27.0482, -0.7831],
      [2.8750, 3.2500, 3.0625, 27.0116, 0.3674],
      [2.8750, 3.0625, 2.9688, 27.0030, -0.1895],
      [2.9688, 3.0625, 3.0156, 27.0007, 0.0933],
      [2.9688, 3.0156, 2.9922, 27.0002, -0.0470],
      [2.9922, 3.0156, 3.0039, 27.0000, 0.0234]]
c = [confere(r.history[:8, 1:6], T2, 4), confere(r.x, 3.0000, 4),
     confere(r.L, 9.2e-5, 6), r.ngev == 17, r.nfev == 0,
     confere(Lgs, 0.0022, 4), round(Lgs / r.L) == 24]
print("  tabela k = 0..7: %s | x* ~ 3.0000: %s | b - a = 9.2e-5: %s | n_g = 17: %s | n_f = 0: %s | L áurea: %s | 24 vezes: %s"
      % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ Exemplo 1
f = lambda x: -2 * np.sin(x) + x**2 / 16
df = lambda x: -2 * np.cos(x) + x / 8
a, b, N = -1, 3, 15
print("\nExemplo 1: f(x) = -2 sin(x) + x^2/16 em [%g,%g], f'(%g) = %.2f < 0, f'(%g) = %.2f > 0, N = %d"
      % (a, b, a, df(a), b, df(b), N))
r = bisection(f, df, a, b, N, verbose=True)
print("Após %d reduções, ponto médio: x* ~ %.4f, f(x*) ~ %.4f, b - a = %.1e" % (N, r.x, r.fx, r.L))
T1 = [[-1.0000, 3.0000, 1.0000, -1.6204, -0.9556],
      [1.0000, 3.0000, 2.0000, -1.5686, 1.0823],
      [1.0000, 2.0000, 1.5000, -1.8544, 0.0460],
      [1.0000, 1.5000, 1.2500, -1.8003, -0.4744],
      [1.2500, 1.5000, 1.3750, -1.8436, -0.2172],
      [1.3750, 1.5000, 1.4375, -1.8531, -0.0861],
      [1.4375, 1.5000, 1.4688, -1.8548, -0.0201],
      [1.4688, 1.5000, 1.4844, -1.8548, 0.0129]]
c = [confere(r.history[:8, 1:6], T1, 4), confere([df(-1), df(3)], [-1.21, 2.35], 2),
     confere(r.x, 1.4783, 4), confere(r.fx, -1.8549, 4), confere(r.L, 1.2e-4, 5)]
print("  tabela k = 0..7: %s | f'(-1), f'(3): %s | x*: %s | f(x*): %s | b - a = 1.2e-4: %s"
      % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------- Para resolver
print("\nPara resolver (15 reduções, ponto médio final):")
f = lambda x: x + 31 / (x + 3); df = lambda x: 1 - 31 / (x + 3)**2
r3 = bisection(f, df, 0, 5, 15)
print("  x + 31/(x+3) em [0,5]:       x* = %.4f, f(x*) = %.4f  (exato: sqrt(31) - 3 = %.4f)"
      % (r3.x, r3.fx, np.sqrt(31) - 3))
f = lambda x: -3 * np.sin(x) + x**2 / 8; df = lambda x: -3 * np.cos(x) + x / 4
r4 = bisection(f, df, 0, 4, 15)
print("  -3 sin(x) + x^2/8 em [0,4]:  x* = %.4f, f(x*) = %.4f, b - a = %.1e" % (r4.x, r4.fx, r4.L))
c = [confere([r3.x, r3.fx], [2.5678, 8.1355], 4), confere(np.sqrt(31) - 3, 2.5678, 4),
     confere([r4.x, r4.fx], [1.4496, -2.7153], 4), confere(r4.L, 1.2e-4, 5)]
print("  1.º: %s | exato: %s | 2.º: %s | b - a = 1.2e-4: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ Quantas iterações?
kk = [int(np.ceil(np.log2(2 / t))) for t in (1e-3, 1e-4)]
print("\nQuantas iterações? [0,2]: log2(2000) = %.2f -> k = %d (tolx = 1e-3); k = %d (tolx = 1e-4)"
      % (np.log2(2000), kk[0], kk[1]))
c = [confere(np.log2(2000), 10.97, 2), kk[0] == 11, kk[1] == 15]
print("  log2 2000 = 10.97: %s | k = 11: %s | k = 15: %s" % simnao(c))
ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 2.4: bisseção

Exemplo 2: f(x) = x^2 + 54/x em [1,4], f'(1) = -52 < 0, f'(4) = 4.625 > 0, N = 15
  k       a_k       b_k       x_k     f(x_k)    df(x_k)
  0    1.0000    4.0000    2.5000    27.8500    -3.6400
  1    2.5000    4.0000    3.2500    27.1779    +1.3876
  2    2.5000    3.2500    2.8750    27.0482    -0.7831
  3    2.8750    3.2500    3.0625    27.0116    +0.3674
  4    2.8750    3.0625    2.9688    27.0030    -0.1895
  5    2.9688    3.0625    3.0156    27.0007    +0.0933
  6    2.9688    3.0156    2.9922    27.0002    -0.0470
  7    2.9922    3.0156    3.0039    27.0000    +0.0234
  8    2.9922    3.0039    2.9980    27.0000    -0.0117
  9    2.9980    3.0039    3.0010    27.0000    +0.0059
 10    2.9980    3.0010    2.9995    27.0000    -0.0029
 11    2.9995    3.0010    3.0002    27.0000    +0.0015
 12    2.9995    3.0002    2.9999    27.0000    -0.0007
 13    2.9999    3.0002    3.0001    27.0000    +0.0004
 14    2.9999    3.0001    3.0000    27.0000 

## 2.5 Método de Newton–Raphson

Exemplo [`ex02_5_newton.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/02_univariate_optimization/02_5_newton/ex02_5_newton.py) · funções da UC: [`newton_1d.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/02_univariate_optimization/02_5_newton/newton_1d.py)

```text
Reproduz os exemplos do deck 2.5 (Newton-Raphson).

Exemplo-guia: f(x) = x^2 + 54/x, x0 = 2.5: x1 = 2.908; 4 iterações até
    x* = 3 (erro 2.4e-12); n_g = n_H = 1 por iteração (+ f'(x0)).
    Paragem |f'| < tolg com tolg = 1e-8 (o slide não fixa tolg; qualquer
    valor entre 1.5e-11 e 1.6e-5 dá as mesmas 4 iterações).
«Convergência quadrática»: x^3 - 2x^2 + 4, x0 = 1, tolg = 1e-3 (3 it.).
«Newton procura f' = 0»: maximizar 8 + 5x - 3x^4 - 2x^6, x0 = 0.5,
    tolg = 1e-7 (5 it.).
«Os quatro métodos»: x^2 + 54/x a partir de x0 = 1 (erro após 5 e 7 it.).

Imprime as tabelas como nos slides, os valores finais e as contagens, e
no fim compara com os slides (vírgula decimal nos slides, ponto aqui).
```

In [5]:
import numpy as np

from newton_1d import newton_1d


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


ok = True
print("Otimização — deck 2.5: Newton-Raphson")

# ------------------------------------------------------- exemplo-guia
f = lambda x: x**2 + 54 / x
df = lambda x: 2 * x - 54 / x**2
ddf = lambda x: 2 + 108 / x**3
x0, tolg, kmax = 2.5, 1e-8, 20
print("\nExemplo-guia: f(x) = x^2 + 54/x, x0 = %g, tolg = %g" % (x0, tolg))
r = newton_1d(f, df, ddf, x0, tolg, kmax)
H = r.history
print("%3s %11s %11s %9s %11s %12s" % ("k", "x_k", "df(x_k)", "ddf(x_k)", "x_k+1", "|x_k+1 - 3|"))
for row in H:
    print("%3d %11.7f %+11.5f %9.4f %11.7f %12.1e" % (row[0], row[1], row[3], row[4], row[5], abs(row[5] - 3)))
print("x* = %.7f, f(x*) = %.4f; %d iterações, n_g = %d, n_H = %d, n_f = %d"
      % (r.x, r.fx, r.nit, r.ngev, r.nhev, r.nfev))
r5 = newton_1d(None, df, ddf, x0, 1e-15, 5)
print("A 5.ª iteração chega à precisão de máquina: x_5 - 3 = %.1e (%d iterações)" % (r5.x - 3, r5.nit))
Tg = np.array([[2.5000000, -3.64000, 8.9120, 2.9084381],
               [2.9084381, -0.56685, 6.3898, 2.9971495],
               [2.9971495, -0.01712, 6.0114, 2.9999973],
               [2.9999973, -0.00002, 6.0000, 3.0000000]])
eg = np.abs(H[:, 5] - 3)
c = [H.shape[0] == 4 and confere(H[:, [1, 5]], Tg[:, [0, 3]], 7) and confere(H[:, 3], Tg[:, 1], 5)
     and confere(H[:, 4], Tg[:, 2], 4),
     confere(eg, [9.2e-2, 2.9e-3, 2.7e-6, 2.4e-12], [3, 4, 7, 13]), confere(H[0, 5], 2.908, 3),
     r.nit == 4, r.ngev == 5, r.nhev == 4, r.nfev == 0, r5.x == 3]
print("  tabela k = 0..3: %s | erros: %s | x1 = 2.908: %s | 4 it.: %s | n_g = 5: %s | n_H = 4: %s | n_f = 0: %s | x_5 = 3: %s"
      % simnao(c))
ok = ok and all(c)

# ------------------------------------------- convergência quadrática
f = lambda x: x**3 - 2 * x**2 + 4
df = lambda x: 3 * x**2 - 4 * x
ddf = lambda x: 6 * x - 4
x0, tolg = 1, 1e-3
print("\nConvergência quadrática: f(x) = x^3 - 2x^2 + 4, x0 = %g, tolg = %g" % (x0, tolg))
r = newton_1d(f, df, ddf, x0, tolg, 20)
H = r.history
print("%3s %9s %10s %9s %9s %11s" % ("k", "x_k", "df(x_k)", "ddf(x_k)", "x_k+1", "df(x_k+1)"))
for row in H:
    print("%3d %9.5f %+10.5f %9.4f %9.5f %+11.5f" % tuple(row[[0, 1, 3, 4, 5, 6]]))
e = np.abs(np.append(H[:, 1], r.x) - 4 / 3)
Hx = ddf(r.x)                        # classificar: 1 avaliação extra de f''
print("|f'(x_3)| = %.1e < tolg: para em x_3 = %.5f (x* = 4/3); f''(x_3) = %.4f > 0 (f''(4/3) = %g): mínimo"
      % (abs(H[-1, 6]), r.x, Hx, ddf(4 / 3)))
print("Erros e_k: %.4f -> %.4f -> %.4f -> %.4f;  e_3/e_2^2 = %.2f (C = 0.75)" % (*e, e[3] / e[2]**2))
Tc = np.array([[1.00000, -1.00000, 2.0000, 1.50000, 0.75000],
               [1.50000, 0.75000, 5.0000, 1.35000, 0.06750],
               [1.35000, 0.06750, 4.1000, 1.33354, 0.00081]])
c = [H.shape[0] == 3 and confere(H[:, [1, 3, 5, 6]], Tc[:, [0, 1, 3, 4]], 5) and confere(H[:, 4], Tc[:, 2], 4),
     confere(abs(H[-1, 6]), 8.1e-4, 5), confere(e, [0.33, 0.17, 0.017, 0.0002], [2, 2, 3, 4]),
     ddf(4 / 3) == 4, Hx > 0, r.ngev == 4, r.nhev == 3]
print("  tabela k = 0..2: %s | |f'(x_3)| = 8.1e-4: %s | erros: %s | f''(4/3) = 4: %s | mínimo: %s | n_g = 4: %s | n_H = 3: %s"
      % simnao(c))
ok = ok and all(c)
r03 = newton_1d(f, df, ddf, 0.3, 1e-8, 20)
print("De x0 = 0.3: x = %.6f, f''(x) = %g < 0: máximo (ponto estacionário errado)" % (r03.x, ddf(r03.x)))
c = [abs(r03.x) < 1e-6, ddf(r03.x) < 0]
print("  converge para 0: %s | máximo: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------ Newton procura f' = 0 (maximizar)
f = lambda x: 8 + 5 * x - 3 * x**4 - 2 * x**6
df = lambda x: 5 - 12 * (x**3 + x**5)
ddf = lambda x: -12 * (3 * x**2 + 5 * x**4)
x0, tolg = 0.5, 1e-7
print("\nMaximizar f(x) = 8 + 5x - 3x^4 - 2x^6, x0 = %g, tolg = %g" % (x0, tolg))
r = newton_1d(f, df, ddf, x0, tolg, 20)
H = r.history
print("%3s %11s %11s %11s %11s %11s" % ("k", "x_k", "f(x_k)", "df(x_k)", "ddf(x_k)", "x_k+1"))
for row in H:
    print("%3d %11.7f %11.6f %+11.6f %11.5f %11.7f" % tuple(row[:6]))
Hx = ddf(r.x)                        # classificar: 1 avaliação extra de f''
print("Após %d iterações |f'| < 1e-7: x* = %.7f, f(x*) = %.6f, f''(x*) = %.4f < 0: máximo"
      % (r.nit, r.x, r.fx, Hx))
Tm = np.array([[0.5000000, 10.281250, 3.125000, -12.75000, 0.7450980],
               [0.7450980, 10.458621, -2.719687, -38.47906, 0.6744184],
               [0.6744184, 10.563259, -0.355311, -28.78702, 0.6620756],
               [0.6620756, 10.565490, -0.009182, -27.30912, 0.6617394],
               [0.6617394, 10.565491, -0.000007, -27.26970, 0.6617391]])
c = [H.shape[0] == 5 and confere(H[:, [1, 5]], Tm[:, [0, 4]], 7) and confere(H[:, 2:4], Tm[:, 1:3], 6)
     and confere(H[:, 4], Tm[:, 3], 5),
     r.nit == 5, confere(r.x, 0.6617391, 7), confere(r.fx, 10.565491, 6), Hx < 0]
print("  tabela k = 0..4: %s | 5 it.: %s | x*: %s | f(x*): %s | máximo: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ os quatro métodos
f = lambda x: x**2 + 54 / x
df = lambda x: 2 * x - 54 / x**2
ddf = lambda x: 2 + 108 / x**3
r5 = newton_1d(f, df, ddf, 1, 1e-2, 20)          # para após 5 it.
r7 = newton_1d(f, df, ddf, 1, 1e-10, 20)         # para após 7 it.
print("\nOs quatro métodos, Newton de x0 = 1: erro após %d it. = %.1e (n_g = %d, n_H = %d); após %d it. = %.1e"
      % (r5.nit, abs(r5.x - 3), r5.ngev, r5.nhev, r7.nit, abs(r7.x - 3)))
# o slide escreve 4,5e-4
c = [r5.nit == 5, round(abs(r5.x - 3) / 1e-5) == 45, r5.ngev == 6, r5.nhev == 5,
     r7.nit == 7, np.floor(np.log10(abs(r7.x - 3))) == -15]
print("  5 it.: %s | erro 4,5e-4: %s | n_g = 6: %s | n_H = 5: %s | 7 it.: %s | erro da ordem de 1e-15: %s"
      % simnao(c))
ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 2.5: Newton-Raphson

Exemplo-guia: f(x) = x^2 + 54/x, x0 = 2.5, tolg = 1e-08
  k         x_k     df(x_k)  ddf(x_k)       x_k+1  |x_k+1 - 3|
  0   2.5000000    -3.64000    8.9120   2.9084381      9.2e-02
  1   2.9084381    -0.56685    6.3898   2.9971495      2.9e-03
  2   2.9971495    -0.01712    6.0114   2.9999973      2.7e-06
  3   2.9999973    -0.00002    6.0000   3.0000000      2.4e-12
x* = 3.0000000, f(x*) = 27.0000; 4 iterações, n_g = 5, n_H = 4, n_f = 0
A 5.ª iteração chega à precisão de máquina: x_5 - 3 = 0.0e+00 (5 iterações)
  tabela k = 0..3: sim | erros: sim | x1 = 2.908: sim | 4 it.: sim | n_g = 5: sim | n_H = 4: sim | n_f = 0: sim | x_5 = 3: sim

Convergência quadrática: f(x) = x^3 - 2x^2 + 4, x0 = 1, tolg = 0.001
  k       x_k    df(x_k)  ddf(x_k)     x_k+1   df(x_k+1)
  0   1.00000   -1.00000    2.0000   1.50000    +0.75000
  1   1.50000   +0.75000    5.0000   1.35000    +0.06750
  2   1.35000   +0.06750    4.1000   1.33354    +0.00081
|f'(x_3)| = 8.1e-

## 2.5 Método de Newton–Raphson salvaguardado

Exemplo [`ex02_5_newton_safeguarded.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/02_univariate_optimization/02_5_newton/ex02_5_newton_safeguarded.py) · funções da UC: [`newton_safeguarded.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/02_univariate_optimization/02_5_newton/newton_safeguarded.py)

```text
Reproduz o exemplo «Newton salvaguardado: bisseção + Newton» do deck 2.5.

f'(x) = arctan x (mínimo de f(x) = x arctan x - ln(1 + x^2)/2 em 0),
intervalo [-1, 2], x0 = 1.5 -- o caso em que Newton puro diverge
(página «Quando falha»).  Um passo de bisseção, depois três de Newton:

  k   x_k        passo de Newton               x_k+1        novo intervalo
  0   1.5        -1.694: fora de (a,b) -> bis.  0.5          [-1; 0.5]
  1   0.5        -0.0796: aceite                -0.0796      [-0.0796; 0.5]
  2   -0.0796    3.4e-4: aceite                 3.4e-4       [-0.0796; 3.4e-4]
  3   3.4e-4     -2.5e-11: aceite               -2.5e-11

Paragem |f'| < tolg = 1e-10 (a do script das figuras).
Imprime a tabela, as contagens e a comparação com Newton puro, e no fim
compara com os slides (vírgula decimal nos slides, ponto aqui).
```

In [6]:
import numpy as np

from newton_safeguarded import newton_safeguarded


def confere_sig(v, s, m):
    """v confere com s mostrado com m algarismos significativos?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    m = np.ravel(np.asarray(m, float))
    e = np.floor(np.log10(np.abs(s)))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (e - m + 1) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


ok = True
print("Otimização — deck 2.5: Newton salvaguardado (bisseção + Newton)")

f = lambda x: x * np.arctan(x) - 0.5 * np.log(1 + x**2)
df = lambda x: np.arctan(x)
ddf = lambda x: 1 / (1 + x**2)

# ---------------------------------------------- Newton puro diverge
x0 = 1.5
print("\nNewton puro para f'(x) = arctan x a partir de x0 = %g (3 iterações):" % x0)
xs = [x0]
for _ in range(3):                    # x_{k+1} = x_k - f'(x_k)/f''(x_k)
    xs.append(xs[-1] - df(xs[-1]) / ddf(xs[-1]))
print("  x_k: " + " -> ".join("%.4f" % v for v in xs) + "  (|x_k| cresce: diverge)")
c = [abs(xs[3]) > abs(xs[2]) > abs(xs[1]) > abs(xs[0])]
print("  diverge: %s" % simnao(c))
ok = ok and all(c)

# ---------------------------------------------- Newton salvaguardado
a, b, tolg = -1.0, 2.0, 1e-10
print("\nNewton salvaguardado em [%g, %g], x0 = %g, tolg = %g:" % (a, b, x0, tolg))
r = newton_safeguarded(f, df, ddf, a, b, x0, tolg, 50, verbose=True)
H = r.history
print("%s; x* ~ %.1e, f(x*) = %.1e" % (r.message, r.x, r.fx))
print("n_g = %d (f'(a), f'(b), f'(x0) e uma por iteração), n_H = %d, n_f = %d" % (r.ngev, r.nhev, r.nfev))
c = [H.shape[0] == 4,
     list(H[:, 3]) == [1, 0, 0, 0],
     confere_sig(H[:, 1], [1.5, 0.5, -0.0796, 3.4e-4], [2, 1, 3, 2]),
     confere_sig(H[:, 2], [-1.694, -0.0796, 3.4e-4, -2.5e-11], [4, 3, 2, 2]),
     confere_sig(H[:, 4], [0.5, -0.0796, 3.4e-4, -2.5e-11], [1, 3, 2, 2]),
     confere_sig(H[:3, 5:7], [[-1, 0.5], [-0.0796, 0.5], [-0.0796, 3.4e-4]], [[1, 1], [3, 1], [3, 2]]),
     r.nbis == 1, r.ngev == 7, r.nhev == 4]
print("  4 iterações: %s | 1 bisseção + 3 Newton: %s | x_k: %s | passos de Newton: %s | x_k+1: %s"
      " | intervalos: %s | 1 bisseção: %s | n_g = 7: %s | n_H = 4: %s" % simnao(c))
ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 2.5: Newton salvaguardado (bisseção + Newton)

Newton puro para f'(x) = arctan x a partir de x0 = 1.5 (3 iterações):
  x_k: 1.5000 -> -1.6941 -> 2.3211 -> -5.1141  (|x_k| cresce: diverge)
  diverge: sim

Newton salvaguardado em [-1, 2], x0 = 1.5, tolg = 1e-10:
  k          x_k     x_Newton      tipo        x_k+1                   [a, b]   df(x_k+1)
  0   1.5000e+00  -1.6941e+00  bisseção   5.0000e-01  [-1.000e+00;  5.000e-01]   +4.64e-01
  1   5.0000e-01  -7.9560e-02    Newton  -7.9560e-02  [-7.956e-02;  5.000e-01]   -7.94e-02
  2  -7.9560e-02   3.3530e-04    Newton   3.3530e-04  [-7.956e-02;  3.353e-04]   +3.35e-04
  3   3.3530e-04  -2.5131e-11    Newton  -2.5131e-11  [-2.513e-11;  3.353e-04]   -2.51e-11
|f'(x)| < tolg em 4 iterações (1 de bisseção); x* ~ -2.5e-11, f(x*) = 6.3e-22
n_g = 7 (f'(a), f'(b), f'(x0) e uma por iteração), n_H = 4, n_f = 0
  4 iterações: sim | 1 bisseção + 3 Newton: sim | x_k: sim | passos de Newton: sim | x_k+1: sim | intervalos: sim | 1 bis

## 2.6 Aproximação numérica de derivadas

Exemplo [`ex02_6_finite_differences.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/02_univariate_optimization/02_6_finite_differences/ex02_6_finite_differences.py) · funções da UC: [`num_derivs.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/02_univariate_optimization/02_6_finite_differences/num_derivs.py), [`newton_numeric.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/02_univariate_optimization/02_6_finite_differences/newton_numeric.py)

```text
Reproduz os exemplos do deck 2.6 (derivadas numéricas; Newton numérico).

Conta à mão: f(x) = x^2 + 54/x em x = 2.5, h = 0.025 (= 0.01|x|):
    progressiva -3.5295, centrada -3.64086, f'' ~ 8.9127; passo de
    Newton x1 = 2.9085 (exato 2.9084); com 6 casas f'' daria 8.9136.
«Então basta fazer h cada vez menor?»: tabela para h = 0.1 ... 1e-8.
Newton numérico de x0 = 1, h = 0.01|x|, tolg = 1e-4: estagna em 3.00010,
    para em k = 6 e devolve x_7; n_f = 7*3 + 1 = 22.

Imprime as tabelas como nos slides, os valores finais e as contagens, e
no fim compara com os slides (vírgula decimal nos slides, ponto aqui).
```

In [7]:
import numpy as np

from num_derivs import num_derivs
from newton_numeric import newton_numeric


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


ok = True
print("Otimização — deck 2.6: derivadas numéricas; Newton numérico")

f = lambda x: x**2 + 54 / x
dfex = lambda x: 2 * x - 54 / x**2        # só para comparar
ddfex = lambda x: 2 + 108 / x**3

# ---------------------------------------------------------- conta à mão
x = 2.5
df, ddf, info = num_derivs(f, x)
print("\nConta à mão: x = %g, h = %g (= 0.01|x|); exatos f' = %.2f, f'' = %.3f"
      % (x, info["h"], dfex(x), ddfex(x)))
print("  f(x-h) = %.8f, f(x) = %.8f, f(x+h) = %.8f   (n_f = %d)"
      % (info["fm"], info["f0"], info["fp"], info["nfev"]))
print("  progressiva %.4f (erro %.2f); regressiva %.4f; centrada %.5f (erro %.0e)"
      % (info["df_fwd"], abs(info["df_fwd"] - dfex(x)), info["df_bwd"], df, abs(df - dfex(x))))
print("  f''_num = %.4f (erro %.0e)" % (ddf, abs(ddf - ddfex(x))))
x1 = x - df / ddf
print("  passo de Newton com estes valores: x1 = %.4f (exato: %.4f)" % (x1, x - dfex(x) / ddfex(x)))
r6 = lambda v: np.round(v * 1e6) / 1e6    # valores com 6 casas
ddf6 = (r6(info["fp"]) - 2 * info["f0"] + r6(info["fm"])) / info["h"]**2
print("  com 6 casas (%.6f; %.6f): f''_num = %.4f (cancelamento)" % (r6(info["fm"]), r6(info["fp"]), ddf6))
c = [confere([info["fm"], info["f0"], info["fp"]], [27.94380682, 27.85, 27.76176361], 8),
     confere(info["df_fwd"], -3.5295, 4), confere(abs(info["df_fwd"] - dfex(x)), 0.11, 2),
     confere(df, -3.64086, 5), confere(abs(df - dfex(x)), 9e-4, 4),
     confere(ddf, 8.9127, 4), confere(x1, 2.9085, 4), confere(x - dfex(x) / ddfex(x), 2.9084, 4),
     confere(ddf6, 8.9136, 4), info["nfev"] == 3]
print("  valores de f: %s | progressiva: %s | erro 0.11: %s | centrada: %s | erro 9e-4: %s | f'': %s | x1: %s | exato: %s | 6 casas: %s | n_f = 3: %s"
      % simnao(c))
ok = ok and all(c)

# ------------------------------------------- h cada vez menor?
print("\nh cada vez menor? (dupla precisão, x = 2.5)")
print("%8s %12s %9s %12s %9s" % ("h", "df central", "erro", "ddf central", "erro"))
hh = [0.1, 0.025, 1e-3, 1e-6, 1e-8]
R = np.zeros((len(hh), 4))
for i, h in enumerate(hh):
    d1, d2, _ = num_derivs(f, x, h)
    R[i] = [d1, abs(d1 - dfex(x)), d2, abs(d2 - ddfex(x))]
    print("%8.3g %12.5f %9.1e %12.4f %9.1e" % (h, *R[i]))
Sd = [-3.65385, -3.64086, -3.64000, -3.64000, -3.64000]
Se1 = [1.4e-2, 9e-4, 1.4e-6, 1e-9, 5e-8]; de1 = [3, 4, 7, 9, 8]
Sdd = [8.9231, 8.9127, 8.9120, 8.9067, 0.0000]
Se2 = [1e-2, 7e-4, 1e-6, 5e-3, 9]; de2 = [2, 4, 6, 3, 0]
c = [confere(R[:, 0], Sd, 5), confere(R[:, 1], Se1, de1), confere(R[:, 2], Sdd, 4),
     confere(R[:, 3], Se2, de2), R[4, 2] == 0]
print("  df: %s | erros df: %s | ddf: %s | erros ddf: %s | ddf = 0 com h = 1e-8: %s" % simnao(c))
ok = ok and all(c)
eps = np.finfo(float).eps
print("  ordens de grandeza do h ótimo: eps^(1/3) = %.0e (df), eps^(1/4) = %.1e (ddf)"
      % (eps ** (1 / 3), eps ** (1 / 4)))
c = [confere(eps ** (1 / 3), 6e-6, 6), confere(eps ** (1 / 4), 1.2e-4, 5)]
print("  eps^(1/3) ~ 6e-6: %s | eps^(1/4) ~ 1.2e-4: %s" % simnao(c))
ok = ok and all(c)

# ---------------------------------------------- Newton numérico, x0 = 1
x0, tolg, kmax = 1, 1e-4, 50
print("\nNewton numérico: f(x) = x^2 + 54/x, x0 = %g, h = max(0.01|x|, 1e-4), tolg = %g" % (x0, tolg))
r = newton_numeric(f, x0, tolg, kmax)
H = r.history
print("%3s %9s %8s %11s %10s %9s" % ("k", "x_k", "h", "df_num", "ddf_num", "x_k+1"))
for row in H:
    print("%3d %9.5f %8.4f %+11.4e %10.4f %9.5f" % tuple(row))
print("Para em k = %d e devolve x_%d = %.5f (estagna: x* = 3), f = %.4f" % (H[-1, 0], r.nit, r.x, r.fx))
print("n_f = %d linhas x 3 = %d, + 1 (f(x_%d)) = %d" % (r.nit, 3 * r.nit, r.nit, r.nfev))
d3, _, _ = num_derivs(f, 3, 0.03)
print("Com h = 0.03, f'_num(3) = %.1e (não é 0)" % d3)
ra = newton_numeric(f, r.x, 1e-3, 1, 1e-4)
print("Reduzindo h para 1e-4|x| no fim, uma iteração: x = %.7f (n_f = %d)" % (ra.x, ra.nfev))
Tn = np.array([[1.00000, 0.0100, -52.0054, 110.0108, 1.47273],
               [1.47273, 0.0147, -21.9541, 35.8141, 2.08573],
               [2.08573, 0.0209, -8.2428, 13.9040, 2.67857],
               [2.67857, 0.0268, -2.1700, 7.6203, 2.96334],
               [2.96334, 0.0296, -0.2233, 6.1507, 2.99965],
               [2.99965, 0.0300, -0.0027, 6.0018, 3.00010],
               [3.00010, 0.0300, -4e-7, 6.0000, 3.00010]])
c = [H.shape[0] == 7 and confere(H[:, [1, 5]], Tn[:, [0, 4]], 5) and confere(H[:, [2, 3, 4]], Tn[:, [1, 2, 3]], 4),
     confere(H[6, 3], -4e-7, 7), H[-1, 0] == 6, r.nit == 7, r.nfev == 22,
     confere(r.x, 3.00010, 5), confere(d3, -6e-4, 4), confere(ra.x, 3, 7)]
print("  tabela k = 0..6: %s | df_6 = -4e-7: %s | para em k = 6: %s | devolve x_7: %s | n_f = 22: %s | x = 3.00010: %s | f'_num(3) = -6e-4: %s | 3.0000000: %s"
      % simnao(c))
ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 2.6: derivadas numéricas; Newton numérico

Conta à mão: x = 2.5, h = 0.025 (= 0.01|x|); exatos f' = -3.64, f'' = 8.912
  f(x-h) = 27.94380682, f(x) = 27.85000000, f(x+h) = 27.76176361   (n_f = 3)
  progressiva -3.5295 (erro 0.11); regressiva -3.7523; centrada -3.64086 (erro 9e-04)
  f''_num = 8.9127 (erro 7e-04)
  passo de Newton com estes valores: x1 = 2.9085 (exato: 2.9084)
  com 6 casas (27.943807; 27.761764): f''_num = 8.9136 (cancelamento)
  valores de f: sim | progressiva: sim | erro 0.11: sim | centrada: sim | erro 9e-4: sim | f'': sim | x1: sim | exato: sim | 6 casas: sim | n_f = 3: sim

h cada vez menor? (dupla precisão, x = 2.5)
       h   df central      erro  ddf central      erro
     0.1     -3.65385   1.4e-02       8.9231   1.1e-02
   0.025     -3.64086   8.6e-04       8.9127   6.9e-04
   0.001     -3.64000   1.4e-06       8.9120   1.1e-06
   1e-06     -3.64000   9.7e-10       8.9067   5.3e-03
   1e-08     -3.64000   5.4e-08       0.0000   8.9e+00
  df: